# 100종목 유니버스 제안 비교와 하이브리드 추천

## tl;dr

- 기존 제안과 Fable 제안은 35개 중 18개가 일치한다.
- Fable안은 AI 인프라와 상대가치 페어가 강하지만 최종 미국 기업 비중이 78%다.
- 기존안은 최종 미국 비중 69%로 지역 분산이 좋지만 일부 실질 중복과 저성장 후보가 있다.
- 하이브리드안은 공통 18개에 차별화된 17개를 더해 최종 미국 73%, 영국 8%, 유럽(영국 제외) 13%, 아시아 6%로 구성한다.


## Context & Methods

현재 65종목과 기존 35종목 제안은 앞선 검증 결과 JSON을 사용한다. Fable 35종목은 사용자 제공 목록을 표준 티커와 경제적 본거지 기준으로 정규화했다. 두 목록의 교집합, 최종 지역·섹터 구성, 현재 종목과의 사업/지배구조 중복, 1,260거래일 학습창을 비교한다.


In [1]:
from pathlib import Path
import importlib.util

ROOT = Path(r'C:\Users\westl\PycharmProjects\pythonProject\venv_vf_new\machine\re_study\c2\ai_port')
SCRIPT = ROOT / 'scripts' / 'build_universe_100_comparison.py'
spec = importlib.util.spec_from_file_location('universe_comparison', SCRIPT)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
results = module.build_results(write_outputs=True)
print('checks_all=', all(results['checks'].values()))
print('headline=', results['headline'])


checks_all= True
headline= {'consensus_names': 18, 'prior_final_us_share': 0.69, 'fable_final_us_share': 0.78, 'hybrid_final_us_share': 0.73, 'hybrid_non_us_share': 0.27, 'hybrid_history_gates': 4}


## Results

### Proposal comparison


In [2]:
for row in results['proposal_summaries']:
    print(row)
print('consensus_count=', len(results['consensus']))
print('consensus=', ', '.join(row['ticker'] for row in results['consensus']))


{'proposal': 'Prior proposal', 'add_names': 35, 'us_add_names': 9, 'non_us_add_names': 26, 'final_us_share': 0.69, 'final_non_us_share': 0.31, 'final_uk_share': 0.09, 'final_europe_ex_uk_share': 0.17, 'final_asia_share': 0.05}
{'proposal': 'Fable proposal', 'add_names': 35, 'us_add_names': 18, 'non_us_add_names': 17, 'final_us_share': 0.78, 'final_non_us_share': 0.22, 'final_uk_share': 0.07, 'final_europe_ex_uk_share': 0.09, 'final_asia_share': 0.06}
{'proposal': 'Hybrid recommendation', 'add_names': 35, 'us_add_names': 13, 'non_us_add_names': 22, 'final_us_share': 0.73, 'final_non_us_share': 0.27, 'final_uk_share': 0.08, 'final_europe_ex_uk_share': 0.13, 'final_asia_share': 0.06}
consensus_count= 18
consensus= 285A JP, ASML NA, AZN LN, BKNG US, HSBA LN, KLAC US, MC FP, NESN SW, NOVOB DC, PWR US, RIO LN, RR/ LN, SAP GR, SHEL LN, SIE GR, SNDK US, SU FP, TMO US


### Hybrid recommendation and sector design


In [3]:
for sector in module.SECTOR_ORDER:
    rows = [row for row in results['hybrid_candidates'] if row['sector'] == sector]
    print(f"{sector} ({len(rows)}): " + ', '.join(row['ticker'] for row in rows))

print('\nFinal sector counts:')
for row in results['sector_comparison']:
    if row['proposal'] == 'Hybrid recommendation':
        print(row['sector'], row['final_names'])


Technology (10): 285A JP, SNDK US, ASML NA, SAP GR, KLAC US, ARM US, ANET US, CDNS US, STM FP, 6857 JP
Industrials (4): SU FP, SIE GR, RR/ LN, PWR US
Financials (4): HSBA LN, ALV GR, PGR US, LSEG LN
Healthcare (5): NOVOB DC, AZN LN, TMO US, BSX US, ROG SW
Consumer Discretionary (3): BKNG US, MC FP, TJX US
Consumer Staples (2): PM US, NESN SW
Energy (2): SHEL LN, WMB US
Utilities (2): CEG US, IBE SM
Real Estate (1): VNA GR
Communication Services (1): SPOT US
Materials (1): RIO LN

Final sector counts:
Technology 34
Industrials 13
Financials 12
Healthcare 11
Consumer Discretionary 7
Consumer Staples 5
Energy 5
Utilities 3
Real Estate 4
Communication Services 3
Materials 3


## Decision Implications

하이브리드안은 종목 스토리보다 기존 65종목 대비 한계 분산효과를 우선한다. 실제 편입 전에는 모든 후보에 동일한 데이터 커버리지·유동성·비용·환율·거래일 달력 검증을 적용하고, 상장 이력이 짧은 네 종목은 마스킹과 거래 자격 게이트를 통과해야 한다.


In [4]:
print('history_gates:')
for row in results['history_gates']:
    print(row['ticker'], row['listing_date'])
print('validation_status=', results['validation_status'])
print('validation_caveat=', results['validation_caveat'])


history_gates:
285A JP 2024-12-18
SNDK US 2025-02-24
ARM US 2023-09-14
CEG US 2022-02-02
validation_status= Share with caveats
validation_caveat= The comparison validates structure, overlap, listing history and implementation gates. It does not rank expected returns because current valuation, liquidity and point-in-time fundamental panels for all 35 candidates were not ingested in this analysis.
